# «Семейный щит»: установка сервера из Google Colab

Блокнот подключается к серверу `188.120.246.123` по SSH с серверов Google и запускает установщик
`shield/deploy/install.sh`. С компьютеров в России SSH до этого сервера часто обрезается фильтрами
операторов, а из сети Google проходит.

1. Меню «Среда выполнения» → «Выполнить все» (Ctrl+F9). На предупреждение «Этот блокнот создан не
   Google» ответьте «Всё равно запустить».
2. Когда появится поле «Пароль root», введите пароль сервера и нажмите Enter. Пароль остаётся только
   в памяти этой сессии Colab и никуда не записывается.
3. Ждите 5–15 минут, ход установки виден ниже. Если закрыть вкладку, установка на сервере всё равно
   доработает.
4. В конце блокнот предложит сменить пароль root. Это стоит сделать: старый пароль пересылался в чате.

In [ ]:
# Настройки
HOST = "188.120.246.123"
PORT = 22
DOMAIN = "familyapp1.ru"
INSTALLER = "https://raw.githubusercontent.com/MorfNNNNN/ubt-factory/claude/cool-gates-wyu9vs/shield/deploy/install.sh"

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paramiko"], check=True)

import base64, getpass, hashlib, io, re, time, urllib.request
import paramiko


def connect(password):
    """SSH to the server as root, retrying while the route is flaky."""
    for attempt in range(1, 7):
        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        try:
            client.connect(HOST, port=PORT, username="root", password=password, timeout=30,
                           banner_timeout=60, auth_timeout=60, look_for_keys=False, allow_agent=False)
            return client
        except paramiko.AuthenticationException:
            client.close()
            raise SystemExit("Пароль не подошёл. Запустите эту ячейку ещё раз и введите пароль root заново.")
        except (OSError, paramiko.SSHException) as e:
            client.close()
            print(f"Попытка {attempt} из 6: сервер не ответил ({type(e).__name__}), повторяю через 15 секунд")
            time.sleep(15)
    raise SystemExit("Сервер не отвечает по SSH. Сообщите об этом Claude.")


def run(client, command, stdin_text=None):
    """Runs a command on the server; returns (exit code, stdout bytes, stderr bytes)."""
    stdin, out, err = client.exec_command(command, timeout=120)
    if stdin_text is not None:
        stdin.write(stdin_text)
    stdin.channel.shutdown_write()
    stdout, stderr = out.read(), err.read()
    return out.channel.recv_exit_status(), stdout, stderr


password = getpass.getpass("Пароль root: ")
ssh = connect(password)
key = ssh.get_transport().get_remote_server_key()
fingerprint = base64.b64encode(hashlib.sha256(key.asbytes()).digest()).decode().rstrip("=")
print(f"Подключено к {HOST}. Отпечаток ключа сервера: {key.get_name()} SHA256:{fingerprint}")

In [ ]:
# Запуск установщика и его журнал
ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")

_, state, _ = run(ssh, 'if [ -f /root/install.pid ] && kill -0 "$(cat /root/install.pid)" 2>/dev/null; then echo running; fi')
if b"running" in state:
    print("Установка уже идёт, показываю её журнал.\n")
else:
    script = urllib.request.urlopen(INSTALLER, timeout=60).read()
    sftp = ssh.open_sftp()
    sftp.putfo(io.BytesIO(script), "/root/install.sh")
    sftp.close()
    # nohup and a pid file: the installer keeps running even if this SSH session drops.
    _, state, err = run(ssh, f"(nohup bash -c 'bash /root/install.sh {DOMAIN}; echo INSTALL_EXIT=$?' "
                             "</dev/null >/root/install.log 2>&1 & echo $! >/root/install.pid); echo started")
    if b"started" not in state:
        raise SystemExit("Не удалось запустить установщик: " + err.decode(errors="replace"))
    print("Установка запущена, это 5–15 минут. Вкладку можно закрыть: установка на сервере доработает сама.\n")

offset, pending, exit_code = 0, b"", None
deadline = time.time() + 45 * 60
while exit_code is None and time.time() < deadline:
    time.sleep(10)
    try:
        _, chunk, _ = run(ssh, f"tail -c +{offset + 1} /root/install.log")
    except Exception:
        ssh = connect(password)
        continue
    offset += len(chunk)
    lines = (pending + chunk).split(b"\n")
    pending = lines.pop()
    for line in lines:
        text = ANSI.sub("", line.decode(errors="replace")).split("\r")[-1]
        if text.startswith("INSTALL_EXIT="):
            exit_code = int(text.split("=", 1)[1])
        else:
            print(text, flush=True)

if exit_code is None:
    print("\nУстановка идёт дольше 45 минут. Запустите эту ячейку ещё раз, чтобы снова смотреть журнал.")
elif exit_code == 0:
    print("\nГотово: сервер установлен.")
else:
    raise SystemExit(f"Установщик остановился с ошибкой (код {exit_code}). Скопируйте последние строки журнала выше и отправьте Claude.")

In [ ]:
# Проверка сайта
for path in ["/health", "/privacy", "/terms"]:
    url = f"https://{DOMAIN}{path}"
    try:
        with urllib.request.urlopen(url, timeout=20) as response:
            print(f"{url} → {response.status}")
    except Exception as e:
        print(f"{url} → не открылся ({e}). Если сертификат ещё выпускается, проверьте через пару минут.")

In [ ]:
# Смена пароля root
new_password = getpass.getpass("Новый пароль root, от 12 символов (Enter — пропустить): ")
if not new_password:
    print("Пароль не изменён. Смените его позже: команда passwd на сервере.")
elif len(new_password) < 12 or "\n" in new_password:
    print("Нужен пароль от 12 символов. Пароль не изменён, запустите ячейку ещё раз.")
elif new_password != getpass.getpass("Повторите новый пароль: "):
    print("Пароли не совпали. Пароль не изменён, запустите ячейку ещё раз.")
else:
    code, _, err = run(ssh, "chpasswd", stdin_text=f"root:{new_password}\n")
    if code == 0:
        password = new_password
        print("Пароль root изменён. Сохраните его в надёжном месте и никому не пересылайте.")
    else:
        print("Не получилось сменить пароль: " + err.decode(errors="replace"))
ssh.close()

Готово. Напишите Claude «готово»: он проверит сайт из российских сетей и продолжит публикацию в RuStore.
Дальше SSH не нужен: сервер сам забирает обновления с GitHub каждые 10 минут.